## 📦 Oppsett og avhengigheter

### 1. Opprett virtuelt miljø

Åpne en terminal i prosjektmappen og kjør:

```bash
# Opprett venv
python3 -m venv .venv
```

### 2. Aktiver miljøet

**macOS/Linux:**
```bash
source .venv/bin/activate
```

**Windows (PowerShell):**
```powershell
.\.venv\Scripts\Activate.ps1
```

**Windows (CMD):**
```cmd
.\.venv\Scripts\activate.bat
```

### 3. Installer pakker

```bash
pip install openai chromadb tiktoken python-dotenv httpx fastmcp rich
```

### 4. Velg kernel i VS Code

1. Klikk på kernel-velgeren øverst til høyre i notebooken
2. Velg **"Select Another Kernel..."** → **"Python Environments..."**
3. Velg **.venv**

### 5. Opprett `.env` fil

Lag en fil som heter `.env` i prosjektmappen med dine Azure OpenAI-innstillinger:

```
AZURE_OPENAI_API_KEY=din-api-nøkkel
AZURE_OPENAI_ENDPOINT=https://din-ressurs.openai.azure.com/
AZURE_OPENAI_CHAT_MODEL=gpt-4o-mini
AZURE_OPENAI_EMBED_MODEL=text-embedding-3-large
AZURE_OPENAI_API_VERSION=2024-02-15-preview
```

In [9]:
import os
from dotenv import load_dotenv

# Last inn miljøvariabler fra .env fil
load_dotenv()

# Sjekk at nødvendige Azure-variabler er satt
required_vars = [
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_CHAT_MODEL",
    "AZURE_OPENAI_EMBED_MODEL"
]

missing = [var for var in required_vars if not os.getenv(var)]

if missing:
    print(f"⚠️  Mangler følgende miljøvariabler i .env:")
    for var in missing:
        print(f"   - {var}")
else:
    print("✅ Alle Azure OpenAI-variabler funnet!")

✅ Alle Azure OpenAI-variabler funnet!


---

# 📅 2024: RAG - Retrieval Augmented Generation

## Hva er RAG?

**RAG** (Retrieval Augmented Generation) er en teknikk som lar språkmodeller svare på spørsmål basert på **dine egne data**, uten å måtte trene modellen på nytt.

### Hvorfor RAG?

Språkmodeller har noen begrensninger:
- 🕐 **Utdatert kunnskap** - Modellen vet kun det den ble trent på
- 🏢 **Mangler bedriftsspesifikk info** - Kjenner ikke dine interne dokumenter
- 🎭 **Hallusinasjoner** - Kan finne på svar når den ikke vet

RAG løser dette ved å **hente relevant kontekst** før modellen genererer svar.

### Hvordan fungerer RAG?

```
┌─────────────────────────────────────────────────────────────────┐
│                        RAG Pipeline                             │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  1. INDEKSERING (offline)                                       │
│     ┌──────────┐    ┌──────────────┐    ┌──────────────┐        │
│     │Dokumenter│ -> │ Chunking     │ -> │ Embeddings   │        │
│     │(PDF,TXT) │    │ (del opp)    │    │ (vektorer)   │        │
│     └──────────┘    └──────────────┘    └──────┬───────┘        │
│                                                 │               │
│                                                 v               │ 
│                                         ┌──────────────┐        │
│                                         │ VektorDB     │        │
│                                         │ (ChromaDB)   │        │
│                                         └──────┬───────┘        │
│                                                │                │
│  2. SPØRRING (runtime)                         │                │
│     ┌──────────┐    ┌──────────────┐           │                │
│     │ Bruker   │ -> │ Embed query  │ ──────────┘                │
│     │ spørsmål │    └──────────────┘      (søk)                 │
│     └──────────┘                            │                   │
│                                             v                   │
│                     ┌──────────────────────────────────┐        │
│                     │ Relevante dokumenter returneres  │        │
│                     └──────────────┬───────────────────┘        │
│                                    │                            │
│  3. GENERERING                     v                            │
│     ┌────────────────────────────────────────────────┐          │
│     │  Prompt = Spørsmål + Kontekst fra dokumenter   │          │
│     └────────────────────────┬───────────────────────┘          │
│                              v                                  │
│                       ┌──────────────┐                          │
│                       │   LLM        │                          │
│                       │              │                          │
│                       └──────┬───────┘                          │
│                              v                                  │
│                       ┌──────────────┐                          │
│                       │    Svar      │                          │
│                       └──────────────┘                          │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

## 🛠️ La oss bygge en enkel RAG!

### Steg 1: Sett opp klienter

In [11]:
from openai import AzureOpenAI
import chromadb

# Sett opp Azure OpenAI klient
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-02-15-preview"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

# Deployment-navn for modellene (fra din .env)
CHAT_MODEL = os.getenv("AZURE_OPENAI_CHAT_MODEL")
EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBED_MODEL")

# Sett opp ChromaDB (vektordatabase)
chroma_client = chromadb.Client()

# Lag (eller hent eksisterende) collection for våre dokumenter
# Slett først hvis den finnes, så vi starter med en tom collection
try:
    chroma_client.delete_collection(name="workshop_docs")
except:
    pass

collection = chroma_client.create_collection(
    name="workshop_docs",
    metadata={"hnsw:space": "cosine"}  # Bruk cosine similarity
)

print("✅ Azure OpenAI og ChromaDB klienter satt opp!")
print(f"   📝 Chat model: {CHAT_MODEL}")
print(f"   🔢 Embedding model: {EMBEDDING_MODEL}")

✅ Azure OpenAI og ChromaDB klienter satt opp!
   📝 Chat model: gpt-5.2-chat
   🔢 Embedding model: text-embedding-3-large


### Steg 2: Last inn og chunk PDF-dokumentet

For å jobbe med PDF-er må vi:
1. **Lese** PDF-filen og ekstrahere tekst
2. **Chunke** teksten - dele den opp i mindre biter som passer i LLM-konteksten
3. **Indeksere** hver chunk med embeddings

La oss først installere en PDF-leser:

In [ ]:
%pip install pymupdf --quiet

In [13]:
import fitz  # PyMuPDF
from pathlib import Path

# Last inn PDF
pdf_path = Path("data/Hafslund Delårsrapport per tredje kvartal 2025.pdf")

# Ekstraher tekst fra alle sider
doc = fitz.open(pdf_path)
full_text = ""
page_count = len(doc)  # Lagre antall sider FØR vi lukker dokumentet

for page_num, page in enumerate(doc):
    full_text += f"\n--- Side {page_num + 1} ---\n"
    full_text += page.get_text()

doc.close()

print(f"📄 Lastet: {pdf_path.name}")
print(f"📊 Antall sider: {page_count}")
print(f"📝 Total tekstlengde: {len(full_text):,} tegn")
print(f"\n🔍 Forhåndsvisning av starten:")
print("-" * 50)
print(full_text[:1000])

📄 Lastet: Hafslund Delårsrapport per tredje kvartal 2025.pdf
📊 Antall sider: 19
📝 Total tekstlengde: 40,336 tegn

🔍 Forhåndsvisning av starten:
--------------------------------------------------

--- Side 1 ---
Delårsrapport          
per tredje kvartal 2025

--- Side 2 ---
Introduksjon
Hafslund er et fornybarkonsern bestående av tre forretningsområder: 
Kraftproduksjon, med Norges nest største vannkraftvirksomhet, 
Fjernvarme, som er Norges største fjernvarmeleverandør og Vekst og 
investeringer, som jobber med etablerte og nye vekstinitiativer innenfor 
den fornybare verdikjeden, i tillegg til å omfatte eierskapet i Eidsiva Energi 
med en eierandel på 50 prosent. Eidsiva Energi eier 100 prosent av Elvia, 
Norges største nettselskap.
Gjennomgående i denne rapporten vises sammenligningstall fra i fjor i 
parentes, med mindre annet er oppgitt. 
Resultat og resultatdrivere per tredje kvartal 2025 
• Hafslund fikk et resultat etter skatt per tredje kvartal 2025 på 2 829 
millioner kroner 

### Steg 3: Chunking - Del opp teksten

**Chunking** er prosessen med å dele opp et dokument i mindre biter. Dette er viktig fordi:
- LLM-er har begrenset kontekstvindu
- Mindre chunks gir mer presise søkeresultater
- Embeddings fungerer best på kortere tekster

Det finnes mange chunking-strategier:
- **Fast størrelse**: Del opp etter antall tegn/ord
- **Setningsbasert**: Del ved setningsgrenser
- **Semantisk**: Del ved tematiske skifter
- **Overlapping**: La chunks overlappe for å bevare kontekst

In [14]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 100) -> list[dict]:
    """
    Del opp tekst i overlappende chunks.
    
    Args:
        text: Teksten som skal chunkes
        chunk_size: Maks antall tegn per chunk
        overlap: Antall tegn som overlapper mellom chunks
    
    Returns:
        Liste med chunk-dictionaries
    """
    chunks = []
    start = 0
    chunk_id = 0
    
    while start < len(text):
        # Finn slutten av chunk
        end = start + chunk_size
        
        # Prøv å kutte ved en naturlig grense (punktum, linjeskift)
        if end < len(text):
            # Søk bakover etter en god kutteposisjon
            for sep in ['\n\n', '\n', '. ', ', ']:
                last_sep = text.rfind(sep, start, end)
                if last_sep > start + chunk_size // 2:
                    end = last_sep + len(sep)
                    break
        
        chunk_text = text[start:end].strip()
        
        if chunk_text:  # Ignorer tomme chunks
            chunks.append({
                "id": f"chunk_{chunk_id}",
                "text": chunk_text,
                "metadata": {
                    "start_char": start,
                    "end_char": end,
                    "chunk_index": chunk_id
                }
            })
            chunk_id += 1
        
        # Flytt startpunkt med overlap
        start = end - overlap
    
    return chunks

# Chunk dokumentet
chunks = chunk_text(full_text, chunk_size=800, overlap=150)

print(f"📦 Antall chunks: {len(chunks)}")
print(f"📏 Gjennomsnittlig chunk-størrelse: {sum(len(c['text']) for c in chunks) // len(chunks)} tegn")

📦 Antall chunks: 68
📏 Gjennomsnittlig chunk-størrelse: 738 tegn


In [15]:
# 🔍 La oss se på noen eksempel-chunks!
print("=" * 70)
print("📄 EKSEMPEL PÅ CHUNKS")
print("=" * 70)

for i, chunk in enumerate(chunks[:3]):  # Vis de 3 første
    print(f"\n🔹 Chunk {i+1} (ID: {chunk['id']})")
    print(f"   Posisjon: tegn {chunk['metadata']['start_char']} - {chunk['metadata']['end_char']}")
    print(f"   Lengde: {len(chunk['text'])} tegn")
    print("-" * 50)
    # Vis teksten med innrykk
    preview = chunk['text'][:400] + "..." if len(chunk['text']) > 400 else chunk['text']
    for line in preview.split('\n'):
        print(f"   {line}")
    print("-" * 50)

📄 EKSEMPEL PÅ CHUNKS

🔹 Chunk 1 (ID: chunk_0)
   Posisjon: tegn 0 - 788
   Lengde: 785 tegn
--------------------------------------------------
   --- Side 1 ---
   Delårsrapport          
   per tredje kvartal 2025
   
   --- Side 2 ---
   Introduksjon
   Hafslund er et fornybarkonsern bestående av tre forretningsområder: 
   Kraftproduksjon, med Norges nest største vannkraftvirksomhet, 
   Fjernvarme, som er Norges største fjernvarmeleverandør og Vekst og 
   investeringer, som jobber med etablerte og nye vekstinitiativer innenfor 
   den fornybare verdikjeden, i tille...
--------------------------------------------------

🔹 Chunk 2 (ID: chunk_1)
   Posisjon: tegn 638 - 1394
   Lengde: 754 tegn
--------------------------------------------------
   dre annet er oppgitt. 
   Resultat og resultatdrivere per tredje kvartal 2025 
   • Hafslund fikk et resultat etter skatt per tredje kvartal 2025 på 2 829 
   millioner kroner (2 651 millioner kroner), som er en økning på 178 
   millioner k

### Steg 4: Lag embeddings og indekser chunks

Nå skal vi lage **embeddings** for hver chunk. En embedding er en numerisk vektor som representerer tekstens betydning. Tekster med lignende innhold får lignende vektorer.

La oss se hvordan en embedding ser ut:

In [16]:
def get_embedding(text: str) -> list[float]:
    """Hent embedding for en tekst via Azure OpenAI."""
    response = client.embeddings.create(
        input=text,
        model=EMBEDDING_MODEL
    )
    return response.data[0].embedding

# 🔍 La oss se hvordan en embedding ser ut!
test_text = chunks[0]["text"][:200]
test_embedding = get_embedding(test_text)

print("🔢 EMBEDDING EKSEMPEL")
print("=" * 60)
print(f"\n📝 Input-tekst ({len(test_text)} tegn):")
print(f"   \"{test_text[:100]}...\"")
print(f"\n🔢 Embedding-vektor:")
print(f"   Dimensjoner: {len(test_embedding)}")
print(f"   Første 10 verdier: {test_embedding[:10]}")
print(f"   Siste 10 verdier: {test_embedding[-10:]}")
print(f"\n📊 Statistikk:")
print(f"   Min: {min(test_embedding):.4f}")
print(f"   Max: {max(test_embedding):.4f}")
print(f"   Gjennomsnitt: {sum(test_embedding)/len(test_embedding):.4f}")

🔢 EMBEDDING EKSEMPEL

📝 Input-tekst (200 tegn):
   "--- Side 1 ---
Delårsrapport          
per tredje kvartal 2025

--- Side 2 ---
Introduksjon
Hafslund..."

🔢 Embedding-vektor:
   Dimensjoner: 3072
   Første 10 verdier: [0.034237269312143326, -0.011769061908125877, -0.010649217292666435, 0.003247191198170185, -0.018730640411376953, -0.009122805669903755, -0.06505081057548523, 0.03169800713658333, -0.04128444194793701, -0.0036840729881078005]
   Siste 10 verdier: [0.007361012976616621, 0.008680574595928192, -0.008473724126815796, 0.01659081131219864, 0.006069982890039682, 0.013965953141450882, -0.003908754792064428, -0.0005478854873217642, -0.012246957048773766, -0.008537919260561466]

📊 Statistikk:
   Min: -0.0773
   Max: 0.1023
   Gjennomsnitt: -0.0005


In [17]:
# Indekser alle chunks i ChromaDB
print("🚀 Indekserer alle chunks...")
print("-" * 50)

for i, chunk in enumerate(chunks):
    embedding = get_embedding(chunk["text"])
    
    collection.add(
        ids=[chunk["id"]],
        embeddings=[embedding],
        documents=[chunk["text"]],
        metadatas=[chunk["metadata"]]
    )
    
    # Vis fremdrift
    if (i + 1) % 10 == 0 or i == len(chunks) - 1:
        print(f"   ✅ Indeksert {i + 1}/{len(chunks)} chunks")

print(f"\n🎉 Ferdig! {collection.count()} chunks er nå søkbare!")

🚀 Indekserer alle chunks...
--------------------------------------------------
   ✅ Indeksert 10/68 chunks
   ✅ Indeksert 10/68 chunks
   ✅ Indeksert 20/68 chunks
   ✅ Indeksert 20/68 chunks
   ✅ Indeksert 30/68 chunks
   ✅ Indeksert 30/68 chunks
   ✅ Indeksert 40/68 chunks
   ✅ Indeksert 40/68 chunks
   ✅ Indeksert 50/68 chunks
   ✅ Indeksert 50/68 chunks
   ✅ Indeksert 60/68 chunks
   ✅ Indeksert 60/68 chunks
   ✅ Indeksert 68/68 chunks

🎉 Ferdig! 68 chunks er nå søkbare!
   ✅ Indeksert 68/68 chunks

🎉 Ferdig! 68 chunks er nå søkbare!


### Steg 5: Søk i dokumentet

Nå kan vi søke etter relevante chunks basert på et spørsmål. Søket fungerer ved at:
1. Spørsmålet konverteres til en embedding
2. Vi finner chunks med lignende embeddings (cosine similarity)
3. De mest relevante chunks returneres

In [19]:
def search_documents(query: str, n_results: int = 3) -> list[dict]:
    """Søk etter relevante dokumenter."""
    query_embedding = get_embedding(query)
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )
    
    return [
        {
            "text": doc,
            "metadata": meta,
            "distance": dist
        }
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        )
    ]

# Test søk
spørsmål = "Hvem er daglig leder i Hafslund?"
resultater = search_documents(spørsmål)

print(f"🔍 Spørsmål: {spørsmål}\n")
print("📚 Relevante dokumenter:")
for i, r in enumerate(resultater, 1):
    print(f"\n{i}. [Score: {1-r['distance']:.2f}] {r['text'][:100]}...")

🔍 Spørsmål: Hvem er daglig leder i Hafslund?

📚 Relevante dokumenter:

1. [Score: 0.63] oner og andre 
endringer har de siste årene bidratt til at konsernledelsen har god 
breddeerfaring o...

2. [Score: 0.59] Dette er et område 
med lite kapasitet i strømnettet. Å forsterke strømnettet er derfor 
nødvendig. ...

3. [Score: 0.57] medførte høy skatt for 2022 som ble betalt i 2023. Vanligvis betales skatt i første halvår, men i 20...


### Steg 6: RAG - Kombiner søk med LLM

Nå setter vi sammen hele RAG-pipelinen: søk + generering.

In [22]:
def rag_query(question: str) -> str:
    """Svar på spørsmål med RAG."""
    
    # 1. Hent relevante dokumenter
    relevant_docs = search_documents(question, n_results=3)
    
    # 2. Bygg kontekst
    context = "\n\n".join([doc["text"] for doc in relevant_docs])
    
    # 3. Lag prompt med kontekst
    system_prompt = """Du er en hjelpsom assistent som svarer på spørsmål basert på den gitte konteksten.
    
Regler:
- Svar KUN basert på informasjonen i konteksten
- Hvis svaret ikke finnes i konteksten, si "Jeg finner ikke informasjon om dette i dokumentene."
- Vær konsis og presis
- Svar på norsk"""
    
    user_prompt = f"""Kontekst:
{context}

Spørsmål: {question}"""
    
    # 4. Kall Azure OpenAI LLM
    response = client.chat.completions.create(
        model=CHAT_MODEL,  # Azure bruker deployment-navn
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
    )
    
    return response.choices[0].message.content

print("✅ RAG-funksjon klar!")

✅ RAG-funksjon klar!


### 🎯 Test RAG!

In [23]:
# Test med spørsmål om delårsrapporten
spørsmål_liste = [
    "Hva var omsetningen i tredje kvartal?",
    "Hvordan har vannkraftproduksjonen vært?",
    "Hva er de viktigste finansielle nøkkeltallene?",
    "Hvem er CEO i Hafslund?"
]

for spørsmål in spørsmål_liste:
    print(f"\n{'='*70}")
    print(f"❓ {spørsmål}")
    print(f"{'='*70}")
    svar = rag_query(spørsmål)
    print(f"\n💬 {svar}")


❓ Hva var omsetningen i tredje kvartal?

💬 Jeg finner ikke informasjon om dette i dokumentene.

❓ Hvordan har vannkraftproduksjonen vært?

💬 Jeg finner ikke informasjon om dette i dokumentene.

❓ Hvordan har vannkraftproduksjonen vært?

💬 Vannkraftproduksjonen har vært lavere enn normalt. Per tredje kvartal 2025 var kraftproduksjonen på rundt 12,8–12,9 TWh, som er 9 prosent lavere enn samme periode i 2024 og 7 prosent lavere enn normalproduksjon. Nedgangen skyldes blant annet betydelig mindre snø enn normalt vinteren 2024/2025, som ga mindre snøsmelting og dermed lavere produksjon, til tross for høy produksjon i første kvartal.

❓ Hva er de viktigste finansielle nøkkeltallene?

💬 Vannkraftproduksjonen har vært lavere enn normalt. Per tredje kvartal 2025 var kraftproduksjonen på rundt 12,8–12,9 TWh, som er 9 prosent lavere enn samme periode i 2024 og 7 prosent lavere enn normalproduksjon. Nedgangen skyldes blant annet betydelig mindre snø enn normalt vinteren 2024/2025, som ga mindre s

## 📝 RAG Oppsummering

**Fordeler med RAG:**
- ✅ Oppdatert informasjon uten re-trening
- ✅ Kontroll over datakilder
- ✅ Reduserer hallusinasjoner
- ✅ Sporbarhet - vet hvilke dokumenter som brukes

**Utfordringer:**
- ⚠️ Kvalitet avhenger av chunking-strategi
- ⚠️ Krever god embedding-modell
- ⚠️ Kan være tregt med store datamengder

---

# 📅 2025: MCP - Model Context Protocol

## Hva er MCP?

**MCP** (Model Context Protocol) er en åpen standard fra Anthropic som lar AI-modeller koble seg til **eksterne verktøy og datakilder** på en standardisert måte.

### Hvorfor MCP?

Før MCP måtte hver AI-integrasjon bygges fra bunnen av. Med MCP får vi:

- 🔌 **Plug-and-play** - Ett grensesnitt for alle verktøy
- 🔄 **Gjenbrukbare komponenter** - Servere kan deles på tvers av prosjekter  
- 🛡️ **Sikkerhet** - Kontrollert tilgang til ressurser
- 🌐 **Økosystem** - Voksende bibliotek av ferdige MCP-servere

### MCP Arkitektur

```
┌─────────────────────────────────────────────────────────────────┐
│                     MCP Arkitektur                              │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   ┌─────────────────────────────────────────────────────────┐   │
│   │                    MCP HOST                             │   │
│   │              (Claude, VS Code, etc.)                    │   │
│   │                                                         │   │
│   │   ┌──────────────────────────────────────────────────┐  │   │
│   │   │                 MCP CLIENT                       │  │   │
│   │   │         (håndterer kommunikasjon)                │  │   │
│   │   └──────────────────────────────────────────────────┘  │   │
│   └─────────────────────────┬───────────────────────────────┘   │
│                             │                                   │
│                             │ JSON-RPC over stdio/SSE           │
│                             │                                   │
│   ┌─────────────────────────┴───────────────────────────────┐   │
│   │                                                         │   │
│   │  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐   │   │
│   │  │ MCP SERVER   │  │ MCP SERVER   │  │ MCP SERVER   │   │   │
│   │  │              │  │              │  │              │   │   │
│   │  │ 📁 Filsystem │  │ 🗄️ Database  │  │ 🌐 API        │   │   │
│   │  │              │  │              │  │              │   │   │
│   │  └──────────────┘  └──────────────┘  └──────────────┘   │   │
│   │                                                         │   │
│   └─────────────────────────────────────────────────────────┘   │
│                                                                 │
│   MCP Servere eksponerer:                                       │
│   • 🔧 Tools    - Funksjoner AI kan kalle                       │
│   • 📚 Resources - Data AI kan lese                             │
│   • 💬 Prompts  - Ferdiglagde prompt-templates                  │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

## 🛠️ Enkel MCP Server!

Vi bruker **FastMCP** - et Python-bibliotek som gjør det enkelt å lage MCP-servere.

### Eksempel: En værmeldings-server med ekte data fra Yr.no

Vi kobler oss på [Yr API (met.no)](https://api.met.no/) for å hente **ekte værdata** i sanntid! Dette er det samme API-et som brukes av yr.no.

> 💡 **Tips:** Det finnes også en ferdig MCP-server for Yr: [yr-mcp-server](https://github.com/AnyContext-ai/yr-mcp-server)

In [30]:
from fastmcp import FastMCP
from datetime import datetime
import httpx

# Opprett MCP server
mcp = FastMCP("Workshop Demo Server")

# Norske byer med koordinater for Yr API
NORWEGIAN_CITIES = {
    "Oslo": {"lat": 59.91, "lon": 10.75},
    "Bergen": {"lat": 60.39, "lon": 5.32},
    "Trondheim": {"lat": 63.43, "lon": 10.39},
    "Tromsø": {"lat": 69.65, "lon": 18.96},
    "Stavanger": {"lat": 58.97, "lon": 5.73},
    "Drammen": {"lat": 59.74, "lon": 10.20},
    "Kristiansand": {"lat": 58.15, "lon": 8.00},
}

# Yr API base URL (gratis, krever kun User-Agent header)
YR_API_BASE = "https://api.met.no/weatherapi/locationforecast/2.0"

print("✅ FastMCP server opprettet!")
print(f"📍 Støttede byer: {', '.join(NORWEGIAN_CITIES.keys())}")

✅ FastMCP server opprettet!
📍 Støttede byer: Oslo, Bergen, Trondheim, Tromsø, Stavanger, Drammen, Kristiansand


### Definer Tools

**Tools** er funksjoner som AI-modellen kan kalle for å utføre handlinger.

In [31]:
async def fetch_yr_weather(latitude: float, longitude: float) -> dict:
    """Hent værdata fra Yr API (met.no)"""
    url = f"{YR_API_BASE}/compact?lat={latitude}&lon={longitude}"
    
    async with httpx.AsyncClient() as client:
        response = await client.get(
            url,
            headers={"User-Agent": "Workshop-Demo/1.0 github.com/workshop"}
        )
        response.raise_for_status()
        return response.json()


def parse_yr_weather(data: dict) -> dict:
    """Parse Yr API response til lesbart format"""
    timeseries = data.get("properties", {}).get("timeseries", [])
    if not timeseries:
        return {"error": "Ingen værdata funnet"}
    
    # Hent nåværende vær (første tidspunkt)
    current = timeseries[0]
    instant = current.get("data", {}).get("instant", {}).get("details", {})
    next_1h = current.get("data", {}).get("next_1_hours", {})
    
    return {
        "temperatur": instant.get("air_temperature"),
        "vind": instant.get("wind_speed"),
        "vindretning": instant.get("wind_from_direction"),
        "luftfuktighet": instant.get("relative_humidity"),
        "skydekke": instant.get("cloud_area_fraction"),
        "symbol": next_1h.get("summary", {}).get("symbol_code", "ukjent"),
        "tidspunkt": current.get("time")
    }


# Definer funksjonene FØRST, så registrer med MCP
async def get_weather(city: str) -> str:
    """
    Hent ekte værmelding fra Yr for en norsk by.
    
    Args:
        city: Navnet på byen (Oslo, Bergen, Trondheim, Tromsø, Stavanger, Drammen, Kristiansand)
    
    Returns:
        Værmelding som tekst med data fra yr.no
    """
    if city not in NORWEGIAN_CITIES:
        return f"Beklager, jeg har ikke koordinater for {city}. Tilgjengelige byer: {', '.join(NORWEGIAN_CITIES.keys())}"
    
    coords = NORWEGIAN_CITIES[city]
    
    try:
        data = await fetch_yr_weather(coords["lat"], coords["lon"])
        weather = parse_yr_weather(data)
        
        # Oversett værsymbol til norsk
        symbol_map = {
            "clearsky_day": "☀️ Klarvær",
            "clearsky_night": "🌙 Klarvær",
            "fair_day": "🌤️ Lettskyet",
            "fair_night": "🌤️ Lettskyet",
            "partlycloudy_day": "⛅ Delvis skyet",
            "partlycloudy_night": "⛅ Delvis skyet",
            "cloudy": "☁️ Overskyet",
            "rain": "🌧️ Regn",
            "lightrain": "🌦️ Lett regn",
            "heavyrain": "🌧️ Kraftig regn",
            "snow": "❄️ Snø",
            "lightsnow": "🌨️ Lett snø",
            "sleet": "🌨️ Sludd",
            "fog": "🌫️ Tåke",
        }
        
        symbol_text = symbol_map.get(weather["symbol"], weather["symbol"])
        
        return (
            f"Været i {city} nå (fra yr.no):\n"
            f"🌡️ Temperatur: {weather['temperatur']}°C\n"
            f"🌤️ Forhold: {symbol_text}\n"
            f"💨 Vind: {weather['vind']} m/s\n"
            f"💧 Luftfuktighet: {weather['luftfuktighet']}%\n"
            f"☁️ Skydekke: {weather['skydekke']}%"
        )
    except Exception as e:
        return f"Kunne ikke hente værdata for {city}: {str(e)}"


def get_current_time() -> str:
    """
    Hent nåværende tid i Norge.
    
    Returns:
        Tidspunkt som tekst
    """
    now = datetime.now()
    return f"Klokken er {now.strftime('%H:%M')} den {now.strftime('%d.%m.%Y')}"


def calculate(expression: str) -> str:
    """
    Beregn et matematisk uttrykk.
    
    Args:
        expression: Matematisk uttrykk (f.eks. "2 + 2", "10 * 5")
    
    Returns:
        Resultatet av beregningen
    """
    try:
        # Kun tillat sikre operasjoner
        allowed_chars = set("0123456789+-*/.() ")
        if not all(c in allowed_chars for c in expression):
            return "Ugyldig uttrykk. Kun tall og +, -, *, /, () er tillatt."
        
        result = eval(expression)
        return f"{expression} = {result}"
    except Exception as e:
        return f"Kunne ikke beregne: {e}"


# Registrer funksjonene med MCP (uten å endre funksjonsobjektene)
mcp.tool(get_weather)
mcp.tool(get_current_time)
mcp.tool(calculate)

print("✅ Tools registrert:")
print("   🔧 get_weather: Hent ekte værmelding fra Yr for en norsk by")
print("   🔧 get_current_time: Hent nåværende tid i Norge")
print("   🔧 calculate: Beregn et matematisk uttrykk")

✅ Tools registrert:
   🔧 get_weather: Hent ekte værmelding fra Yr for en norsk by
   🔧 get_current_time: Hent nåværende tid i Norge
   🔧 calculate: Beregn et matematisk uttrykk


### Definer Resources

**Resources** er data som AI-modellen kan lese, lignende filer eller dokumenter.

In [32]:
@mcp.resource("config://settings")
def get_settings() -> str:
    """
    Hent systeminnstillinger.
    """
    return """
    Systeminnstillinger:
    - Språk: Norsk
    - Tidssone: Europe/Oslo
    - Temperaturenhet: Celsius
    - Versjon: 1.0.0
    """

@mcp.resource("docs://help")
def get_help() -> str:
    """
    Hent hjelpedokumentasjon.
    """
    return """
    # Hjelpeguide
    
    ## Tilgjengelige kommandoer:
    
    1. **get_weather(city)** - Hent værmelding for en by
    2. **get_current_time()** - Hent nåværende tid
    3. **calculate(expression)** - Beregn matematikk
    
    ## Støttede byer:
    Oslo, Bergen, Trondheim, Tromsø
    """

print("✅ Resources registrert!")

✅ Resources registrert!


### Test tools direkte

In [33]:
# Test våre tools (må bruke asyncio for async funksjoner)
import asyncio

async def test_tools():
    print("🔧 Testing tools:\n")
    
    print("1. Værmelding fra Yr.no:")
    print(await get_weather('Oslo'))
    print()
    print(await get_weather('Bergen'))
    
    print("\n2. Tid:")
    print(f"   {get_current_time()}")
    
    print("\n3. Kalkulator:")
    print(f"   {calculate('2 + 2')}")
    print(f"   {calculate('100 / 4 * 3')}")

# Kjør async test
await test_tools()

🔧 Testing tools:

1. Værmelding fra Yr.no:
Været i Oslo nå (fra yr.no):
🌡️ Temperatur: -10.8°C
🌤️ Forhold: ☁️ Overskyet
💨 Vind: 0.9 m/s
💧 Luftfuktighet: 55.7%
☁️ Skydekke: 100.0%

Været i Bergen nå (fra yr.no):
🌡️ Temperatur: 2.6°C
🌤️ Forhold: ☁️ Overskyet
💨 Vind: 10.0 m/s
💧 Luftfuktighet: 55.5%
☁️ Skydekke: 100.0%

2. Tid:
   Klokken er 16:34 den 11.01.2026

3. Kalkulator:
   2 + 2 = 4
   100 / 4 * 3 = 75.0
Været i Oslo nå (fra yr.no):
🌡️ Temperatur: -10.8°C
🌤️ Forhold: ☁️ Overskyet
💨 Vind: 0.9 m/s
💧 Luftfuktighet: 55.7%
☁️ Skydekke: 100.0%

Været i Bergen nå (fra yr.no):
🌡️ Temperatur: 2.6°C
🌤️ Forhold: ☁️ Overskyet
💨 Vind: 10.0 m/s
💧 Luftfuktighet: 55.5%
☁️ Skydekke: 100.0%

2. Tid:
   Klokken er 16:34 den 11.01.2026

3. Kalkulator:
   2 + 2 = 4
   100 / 4 * 3 = 75.0


### Simuler MCP Tool-calling med LLM

Her viser vi hvordan en LLM kan bruke tools via function calling.

In [34]:
import json

# Definer tools for OpenAI (med ekte Yr-integrasjon)
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Hent ekte værmelding fra yr.no for en norsk by (Oslo, Bergen, Trondheim, Tromsø, Stavanger, Drammen, Kristiansand)",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Navnet på byen"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Hent nåværende tid i Norge",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Beregn et matematisk uttrykk",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Matematisk uttrykk"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

# Map tool navn til funksjoner (get_weather er async!)
tool_functions = {
    "get_weather": get_weather,  # async funksjon
    "get_current_time": get_current_time,
    "calculate": calculate
}

print("✅ Tools definert for LLM!")
print("🌦️ Værdata hentes nå LIVE fra yr.no!")

✅ Tools definert for LLM!
🌦️ Værdata hentes nå LIVE fra yr.no!


In [35]:
import asyncio
import inspect

async def chat_with_tools(user_message: str) -> str:
    """
    Chat med LLM som har tilgang til tools (inkludert async tools).
    """
    messages = [
        {
            "role": "system", 
            "content": "Du er en hjelpsom assistent. Bruk de tilgjengelige verktøyene for å svare på spørsmål. Svar alltid på norsk."
        },
        {"role": "user", "content": user_message}
    ]
    
    # Første kall til Azure OpenAI
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    assistant_message = response.choices[0].message
    
    # Sjekk om LLM vil bruke tools
    if assistant_message.tool_calls:
        messages.append(assistant_message)
        
        # Utfør hver tool call
        for tool_call in assistant_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            print(f"🔧 Kaller: {function_name}({function_args})")
            
            # Kall funksjonen (håndter både sync og async)
            func = tool_functions[function_name]
            if inspect.iscoroutinefunction(func):
                function_result = await func(**function_args)
            else:
                function_result = func(**function_args)
            
            print(f"   ➡️  Resultat: {function_result[:100]}..." if len(str(function_result)) > 100 else f"   ➡️  Resultat: {function_result}")
            
            # Legg til resultatet i meldinger
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(function_result)
            })
        
        # Få endelig svar fra LLM
        final_response = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=messages
        )
        
        return final_response.choices[0].message.content
    
    return assistant_message.content

print("✅ Chat-funksjon med tools klar!")

✅ Chat-funksjon med tools klar!


### 🎯 Test MCP-lignende Tool Calling!

In [36]:
# Test med ulike spørsmål - nå med EKTE værdata fra Yr!
spørsmål_liste = [
    "Hva er været i Oslo og Bergen akkurat nå?",
    "Hva er klokken nå?",
    "Hva er 15% av 850?",
    "Sammenlign været i Tromsø og Trondheim - hvor er det kaldest?"
]

for spørsmål in spørsmål_liste:
    print(f"\n{'='*60}")
    print(f"❓ {spørsmål}")
    print(f"{'='*60}")
    svar = await chat_with_tools(spørsmål)
    print(f"\n💬 {svar}")


❓ Hva er været i Oslo og Bergen akkurat nå?
🔧 Kaller: get_weather({'city': 'Oslo'})
   ➡️  Resultat: Været i Oslo nå (fra yr.no):
🌡️ Temperatur: -10.8°C
🌤️ Forhold: ☁️ Overskyet
💨 Vind: 0.9 m/s
💧 Luftf...
🔧 Kaller: get_weather({'city': 'Bergen'})
🔧 Kaller: get_weather({'city': 'Oslo'})
   ➡️  Resultat: Været i Oslo nå (fra yr.no):
🌡️ Temperatur: -10.8°C
🌤️ Forhold: ☁️ Overskyet
💨 Vind: 0.9 m/s
💧 Luftf...
🔧 Kaller: get_weather({'city': 'Bergen'})
   ➡️  Resultat: Været i Bergen nå (fra yr.no):
🌡️ Temperatur: 2.6°C
🌤️ Forhold: ☁️ Overskyet
💨 Vind: 10.0 m/s
💧 Luft...
   ➡️  Resultat: Været i Bergen nå (fra yr.no):
🌡️ Temperatur: 2.6°C
🌤️ Forhold: ☁️ Overskyet
💨 Vind: 10.0 m/s
💧 Luft...

💬 Her er været akkurat nå:

**🌆 Oslo**
- 🌡️ Temperatur: **–10,8 °C**
- ☁️ Forhold: **Overskyet**
- 💨 Vind: **0,9 m/s**
- 💧 Luftfuktighet: **55,7 %**
- ☁️ Skydekke: **100 %**

**🌧️ Bergen**
- 🌡️ Temperatur: **2,6 °C**
- ☁️ Forhold: **Overskyet**
- 💨 Vind: **10,0 m/s**
- 💧 Luftfuktighet: **55,5 %**
- ☁️ Sky

### 💬 Interaktiv Chat i Notebooken

La oss lage en enkel chat-løkke som kjører direkte her i notebooken!

In [ ]:
# 💬 Interaktiv chat direkte i notebooken!
# Kjør denne cellen og skriv spørsmål - skriv 'quit' for å avslutte

async def interactive_chat():
    """Enkel interaktiv chat-løkke."""
    print("=" * 60)
    print("🤖 MCP Chat Demo - Skriv 'quit' for å avslutte")
    print("=" * 60)
    print("\n💡 Prøv: 'Hva er været i Oslo?', 'Hva er 20% av 500?'\n")
    
    while True:
        # Hent input fra bruker
        try:
            user_input = input("Du: ")
        except EOFError:
            break
            
        if user_input.lower() in ['quit', 'exit', 'q']:
            print("\n👋 Ha det!")
            break
        
        if not user_input.strip():
            continue
        
        print("\n⏳ Tenker...\n")
        
        try:
            response = await chat_with_tools(user_input)
            print(f"\n🤖 Assistent: {response}\n")
            print("-" * 60)
        except Exception as e:
            print(f"\n❌ Feil: {e}\n")

# Start chatten
await interactive_chat()

🤖 MCP Chat Demo - Skriv 'quit' for å avslutte

💡 Prøv: 'Hva er været i Oslo?', 'Hva er 20% av 500?'


⏳ Tenker...


⏳ Tenker...

🔧 Kaller: get_weather({'city': 'Oslo'})
   ➡️  Resultat: Været i Oslo nå (fra yr.no):
🌡️ Temperatur: -10.8°C
🌤️ Forhold: ☁️ Overskyet
💨 Vind: 0.9 m/s
💧 Luftf...
🔧 Kaller: get_weather({'city': 'Oslo'})
   ➡️  Resultat: Været i Oslo nå (fra yr.no):
🌡️ Temperatur: -10.8°C
🌤️ Forhold: ☁️ Overskyet
💨 Vind: 0.9 m/s
💧 Luftf...

🤖 Assistent: Skøyen ligger i Oslo, og akkurat nå er været slik der:

🌡️ **Temperatur:** rundt **–11 °C**  
☁️ **Forhold:** **Overskyet**  
💨 **Vind:** svak vind, ca. **1 m/s**  
💧 **Luftfuktighet:** ca. **56 %**

Det er altså kaldt, men rolig vær. Kle deg godt hvis du skal ut 😊

------------------------------------------------------------

🤖 Assistent: Skøyen ligger i Oslo, og akkurat nå er været slik der:

🌡️ **Temperatur:** rundt **–11 °C**  
☁️ **Forhold:** **Overskyet**  
💨 **Vind:** svak vind, ca. **1 m/s**  
💧 **Luftfuktighet:** ca. **5

## 📝 MCP Oppsummering

**Fordeler med MCP:**
- ✅ Standardisert protokoll for AI-verktøy
- ✅ Separasjon mellom AI og verktøy
- ✅ Sikker og kontrollert tilgang
- ✅ Voksende økosystem av ferdige servere

**Populære MCP-servere:**
- 📁 Filesystem - Les/skriv filer
- 🗄️ Database - PostgreSQL, SQLite
- 🌐 Web - Fetch, Puppeteer
- 🔧 DevTools - Git, GitHub, Slack

---

### 🖥️ Bonus: Streamlit Chat-klient

Vi har også laget en enkel Streamlit-frontend som fungerer som en MCP-klient!

**Installer Streamlit:**
```bash
pip install streamlit
```

**Kjør applikasjonen:**
```bash
streamlit run mcp_client_app.py
```

Dette åpner en chat-applikasjon i nettleseren der du kan:
- 💬 Chatte med AI-assistenten
- 🌦️ Spørre om været i norske byer (live fra yr.no)
- 🧮 Gjøre beregninger
- ⏰ Spørre om tiden
- 🔧 Se hvilke tools som ble brukt bak kulissene

# 🎓 Oppsummering

## Hva vi har lært:

| År | Teknologi | Hovedkonsept | Bruksområde |
|------|-----------|--------------|-------------|
| 2024 | **RAG** | Hent kontekst → Generer svar | Snakk med egne dokumenter |
| 2025 | **MCP** | Standardisert verktøy-protokoll | Koble AI til API-er og systemer |

## Neste steg:

1. **Eksperimenter med RAG** på egne dokumenter
2. **Bygg en MCP-server** for interne systemer
3. **Kombiner RAG + MCP** for kraftige AI-assistenter

## Ressurser:

- 📖 [LangChain RAG Tutorial](https://python.langchain.com/docs/tutorials/rag/)
- 📖 [MCP Dokumentasjon](https://modelcontextprotocol.io/)
- 📖 [FastMCP GitHub](https://github.com/jlowin/fastmcp)
- 📖 [ChromaDB Docs](https://docs.trychroma.com/)

---

**Spørsmål? 🙋‍♂️**